# 06 – Métricas do Lote 5 (L5), por célula

ESP-02, ESP-03, ESP-06, QA-05, ADD-05 e ADD-06, segundo `spec/metodologia_metricas.md` (secções 7E e 7F), com `cell` (grelha `GRID_DEG = 0.01`, cerca de 1 km) no lugar de `country`.

- Lê `processed/<sensor>_<ano>.parquet` (notebook 01), **um ficheiro de cada vez e só com as colunas necessárias** (inclui `latitude`/`longitude`), e as tabelas do L1 (notebook 02): `l1_cal_coverage`, `l1_meta_month_validity`, `l1_meta_year_validity`, `l1_cal_annual` e `l1_cal_monthly` (ou `l1_stats_cal_monthly` se `STATS_GRID_DEG` ≠ `GRID_DEG`).
- Por ficheiro guarda as células-dia elegíveis e o ADD-05 em `metrics/_partial_lote5/` e **retoma** se a sessão cair (`FORCE = True` refaz). A retoma é por marcador (`_done.json`) com uma assinatura que inclui a **grelha**: mudar `GRID_DEG` invalida os parciais. Por ano guarda o `cell_year` (dias com deteção por célula); o ADD-06 também fica em parciais por fatia.
- O fatiamento por blocos de países desapareceu. As tabelas por célula são **esparsas** (só células com atividade): `l5_cell_year` (célula, série, ano, `n_days`, a base da recorrência), `l5_esp06_cell_years_active` (anos válidos com atividade por célula), `l5_add05_type_sensitivity` e `l5_add06_union_step` (por célula, mais uma linha global `cell = -1`).
- ESP-02, ESP-03 e ESP-06 são estatísticas de **população** (`cell = -1`) e calculam-se **sobre todas as células de `cell_year` de uma vez** (médias, medianas e percentis não se somam entre fatias). O `cell_year` consolidado cabe em memória; se faltar RAM ative *High-RAM*.
- ADD-05: a linha global recalcula-se a partir das somas das células (o delta não se soma). ADD-06: faz-se por **fatias de `CHUNK_CELLS` células** (por faixa de latitude), densificando só a fatia; a linha global soma `sum_modis`/`sum_combined` entre fatias e recalcula o rácio (`n_months` é a união de meses, não se soma).
- QA-05 é só um **proxy global** (dia com algum registo no sensor); não é cobertura real por região.
- Consolida com DuckDB (fora da memória) em pastas Parquet `metrics/l5_*/` e escreve `_runs` (linha `lote5`) **por último**: uma execução que pára a meio não fica registada.
- O código vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`.

**Antes de correr:** o atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`) e os notebooks 01 e 02 têm de ter corrido (com a mesma `GRID_DEG`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # adjust if the Drive shortcut has a different name
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')
REPO_BRANCH = 'metricas-pipeline'


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'switch', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))
METRICS.mkdir(exist_ok=True)

In [ ]:
# cell: params
# For a short run: SENSORS = {'modis'}; YEARS = {2018}
SENSORS = {'modis', 'viirs-jpss1'}  # None selects the same core cohort; S-NPP uses notebook 09
YEARS = None            # None selects all available years of the selected sensors
FORCE = False           # Rebuild selected caches even when their provenance matches

# --- grelha e volume: a mesma do notebook 02 ---
GRID_DEG = 0.01                # lado da celula das tabelas de base (cell_year, ESP-*, ADD-05), em graus
STATS_GRID_DEG = GRID_DEG      # grelha de onde o ADD-06 le o calendario mensal (l1_stats_cal_monthly se for diferente)
MAX_UNKNOWN_DAYS = 0           # meses/anos validos: no maximo estes dias de cobertura desconhecida
CHUNK_CELLS = 20000            # celulas por fatia do ADD-06 (densifica-se uma fatia de cada vez)

In [ ]:
# cell: defs
import hashlib, json, re, shutil, subprocess, tempfile, time, datetime
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

from metrics import common
from metrics.calendar_metrics import densify_monthly, eligible_cell_days
from metrics.cell_recurrence import (
    build_cell_year, esp02_active_cells, esp03_recurrence, esp06_cell_years_active, esp06_interannual,
)
from metrics.filter_sensitivity import add05_type_sensitivity
from metrics.coverage_steps import add06_union_step, qa05_confirmed_days
from metrics.store import ParquetSink, default_partitions, prepare_table, stats_source, write_run, write_table

assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
common._grid_shape(GRID_DEG)
common._grid_shape(STATS_GRID_DEG)
common.coarsen_cell(np.array([0]), GRID_DEG, STATS_GRID_DEG)     # ValueError se nao for multiplo
if int(CHUNK_CELLS) < 1:
    raise ValueError('CHUNK_CELLS tem de ser >= 1')

SINK = ParquetSink(METRICS)
STATS_MONTHLY = stats_source('l1_cal_monthly', GRID_DEG, STATS_GRID_DEG)
REQUIRED = ['l1_cal_coverage', 'l1_meta_month_validity', 'l1_meta_year_validity', 'l1_cal_annual', STATS_MONTHLY]
missing = [n for n in REQUIRED if not SINK.exists(n)]
if missing:
    raise FileNotFoundError(f'Faltam em {METRICS}: {missing}. Corra primeiro o notebook 02_metrics_lote1 '
                            f'(com GRID_DEG = {GRID_DEG} e STATS_GRID_DEG = {STATS_GRID_DEG}).')

COLS = ['instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})\.parquet$', re.I)
CORE_SENSORS = {'modis', 'viirs-jpss1'}
CACHE_SCHEMA = 3
FILE_TABLES = ['cell_days', 'add05', 'add05_global']
PART = METRICS / '_partial_lote5'
PART.mkdir(parents=True, exist_ok=True)
(PART / 'add06').mkdir(exist_ok=True)


# --- parciais com escrita atomica e marcadores de proveniencia ---
def atomic_part(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(value, path):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    tmp.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def source_signature(sensor, year, path):
    stat = path.stat()
    return dict(schema=CACHE_SCHEMA, sensor=sensor, year=year, path=str(path.resolve()),
                size=stat.st_size, mtime_ns=stat.st_mtime_ns, grid_deg=GRID_DEG,
                confidence=list(common.ELIGIBLE_CONFIDENCE), eligible_type=common.ELIGIBLE_TYPE)


def is_done(sensor, year, path):
    out = PART / f'{sensor}_{year}'
    info = read_json(out / '_done.json')
    return (isinstance(info, dict) and info.get('signature') == source_signature(sensor, year, path)
            and all((out / f'{name}.parquet').is_file() for name in FILE_TABLES))


def select_sensors():
    selected = CORE_SENSORS if SENSORS is None else set(SENSORS)
    unsupported = selected - CORE_SENSORS
    if unsupported:
        raise ValueError(f'nenhum ficheiro selecionado: unsupported sensors {sorted(unsupported)}; '
                         f'core sensors: {sorted(CORE_SENSORS)}. Use 09_har12_snpp.ipynb for S-NPP.')
    return selected


def index_core_files():
    found = {}
    for p in sorted(PROCESSED.glob('*.parquet')):
        m = FILE_RE.match(p.name)
        if m and m['sensor'].lower() in CORE_SENSORS:
            found[(m['sensor'].lower(), int(m['year']))] = p
    return found


def process_file(sensor, year, path):
    """Calcula as celulas-dia elegiveis e o ADD-05 (por celula) de UM ficheiro; o marcador escreve-se no fim."""
    out = PART / f'{sensor}_{year}'
    out.mkdir(exist_ok=True)
    # Um marcador invalido nao pode anunciar uma cache a meio como completa.
    (out / '_done.json').unlink(missing_ok=True)
    t0 = time.time()
    df = pd.read_parquet(path, columns=COLS)
    expected = 'MODIS' if sensor == 'modis' else 'VIIRS'
    if not df['instrument'].astype('string').str.upper().eq(expected).fillna(False).all():
        raise ValueError(f'{path.name}: instrument does not match {sensor}')
    if sensor == 'viirs-jpss1':
        satellite = df['satellite'].astype('string').str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
        if not satellite.isin(['N20', 'NOAA20', 'J1', '1']).all():
            raise ValueError(f'{path.name}: expected NOAA-20; S-NPP belongs in 09_har12_snpp.ipynb')
    info = dict(sensor=sensor, year=year, grid_deg=GRID_DEG, n_rows=len(df))
    cell_days = eligible_cell_days(df, GRID_DEG)
    add05 = add05_type_sensitivity(df, grid_deg=GRID_DEG)
    is_global = (add05['cell'] == common.GLOBAL_CELL).to_numpy()
    atomic_part(cell_days, out / 'cell_days.parquet')
    # por celula (com colunas de celula) e, a parte, as linhas globais deste ficheiro (so para a verificacao)
    atomic_part(prepare_table(add05[~is_global], GRID_DEG, ['year', 'instrument', 'variant']), out / 'add05.parquet')
    atomic_part(add05[is_global].reset_index(drop=True), out / 'add05_global.parquet')
    info.update(n_cell_days=len(cell_days), seconds=round(time.time() - t0, 1),
                signature=source_signature(sensor, year, path))
    atomic_json(info, out / '_done.json')
    return info


def year_status(year, files, todo):
    """Only current sources count; combined requires every available core source in this year."""
    every = {s for (s, y) in files if y == year}
    done = [s for s, y in todo if y == year and is_done(s, y, files[(s, y)])]
    series = set()
    if 'modis' in done:
        series.add('modis')
    if 'viirs-jpss1' in done:
        series.add('viirs')
    if every and set(done) == every:
        series.add('combined')
    return done, series


# --- DuckDB: consultas e consolidacao fora da memoria ---
def duckdb_list(files):
    return '[' + ', '.join("'" + str(f).replace('\\', '/').replace("'", "''") + "'" for f in files) + ']'


def duckdb_connect():
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{Path(tempfile.gettempdir()).as_posix()}/duckdb_spill'")
    return con


def table_source(name):
    pattern = (SINK.path(name) / '**' / '*.parquet').as_posix().replace("'", "''")
    return f"read_parquet('{pattern}', hive_partitioning=true, union_by_name=true)"


def query(sql):
    con = duckdb_connect()
    try:
        return con.execute(sql).df()
    finally:
        con.close()


def table_fingerprint(name):
    """Impressao digital dos ficheiros de uma tabela publicada (caminho, tamanho, mtime)."""
    root = SINK.path(name)
    items = sorted((p.relative_to(root).as_posix(), p.stat().st_size, p.stat().st_mtime_ns)
                   for p in root.rglob('*.parquet'))
    return hashlib.sha1(json.dumps(items).encode()).hexdigest()


def consolidate(name, files, keys, global_rows=None, check_keys=True):
    """Junta os parciais `files` numa tabela publicada (pasta Parquet ordenada por (ilat, ilon) e
    particionada por series/lat_band) e devolve o numero de linhas. Verifica chaves repetidas.
    `global_rows` (opcional): linhas `cell = -1` ja prontas (com colunas de celula); nesse caso as
    linhas `cell < 0` dos parciais sao ignoradas. Com linhas globais nao ha particao por lat_band."""
    files = list(files)
    source = f'read_parquet({duckdb_list(files)}, union_by_name=true)'
    schema = pd.read_parquet(files[0]).iloc[:0]
    partitions = default_partitions(schema)
    if global_rows is not None:
        gfile = PART / f'{name}_global.parquet'
        atomic_part(global_rows, gfile)
        source = (f'(SELECT * FROM {source} WHERE cell >= 0 UNION ALL BY NAME '
                  f'SELECT * FROM read_parquet({duckdb_list([gfile])}))')
        if len(global_rows):
            partitions = [p for p in partitions if p != 'lat_band']
    has_cell = 'cell' in schema.columns
    order = (['ilat NULLS FIRST', 'ilon NULLS FIRST'] + [k for k in keys if k != 'cell']) if has_cell else list(keys)
    con = duckdb_connect()
    try:
        n_rows = con.execute(f'SELECT count(*) FROM {source}').fetchone()[0]
        if check_keys and n_rows:
            repeated = con.execute(
                f'SELECT count(*) FROM (SELECT 1 FROM {source} GROUP BY {", ".join(keys)} HAVING count(*) > 1)'
            ).fetchone()[0]
            assert repeated == 0, f'{name}: {repeated} chaves repetidas entre ficheiros (datas fora do ano do ficheiro?)'
        if n_rows == 0:
            SINK.write(name, schema, ())
            return 0
        tmp = METRICS / f'{name}.tmp'
        if tmp.exists():
            shutil.rmtree(tmp)
        if partitions:
            part, target = f', PARTITION_BY ({", ".join(partitions)})', tmp
        else:       # sem particoes o DuckDB escreve um ficheiro: po-lo dentro da pasta da tabela
            tmp.mkdir(parents=True)
            part, target = '', tmp / 'part-0.parquet'
        destination = target.as_posix().replace("'", "''")
        con.execute(f"COPY (SELECT * FROM {source} ORDER BY {', '.join(order)}) TO '{destination}' "
                    f"(FORMAT PARQUET, COMPRESSION zstd{part}, OVERWRITE_OR_IGNORE)")
    finally:
        con.close()
    SINK._publish(tmp, name)
    return int(n_rows)


# --- ADD-05 global: recalculado a partir das somas das celulas (o delta nao se soma) ---
def add05_global_rows(files):
    glob = query(f"SELECT year, instrument, variant, sum(n_eligible)::BIGINT AS n_eligible, "
                 f"sum(n_cell_days)::BIGINT AS n_cell_days FROM read_parquet({duckdb_list(files)}, union_by_name=true) "
                 f"GROUP BY year, instrument, variant ORDER BY year, instrument, variant")
    base = (glob[glob['variant'] == 'default'][['year', 'instrument', 'n_cell_days']]
            .rename(columns={'n_cell_days': 'n_default'}))
    glob = glob.merge(base, on=['year', 'instrument'], how='left')
    n_default = glob['n_default'].where(glob['n_default'] != 0)
    glob['delta_cell_days_pct'] = (100.0 * (glob['n_cell_days'] - glob['n_default']) / n_default).astype('float64')
    glob['cell'] = np.full(len(glob), common.GLOBAL_CELL, dtype='int64')
    glob['year'] = glob['year'].astype('int64')
    cols = ['cell', 'year', 'instrument', 'variant', 'n_eligible', 'n_cell_days', 'delta_cell_days_pct']
    return prepare_table(glob[cols], GRID_DEG, ['year', 'instrument', 'variant'])


# --- ADD-06 por fatias de celulas (por faixa de latitude) ---
ADD06_SERIES = ('modis', 'combined')


def empty_add06():
    monthly = pd.DataFrame({c: pd.Series(dtype=t) for c, t in dict(
        cell='int64', year='int64', month='int64', series='object', cell_days='int64',
        partial='bool', n_unknown_days='int64').items()})
    return add06_union_step(monthly, MAX_UNKNOWN_DAYS)


def add06_condition(comp):
    """Predicado SQL sobre (series, year) dos pares calculados que o ADD-06 usa (None se nao houver nenhum)."""
    terms = []
    for series in ADD06_SERIES:
        ys = sorted(int(y) for y in comp.loc[comp['series'] == series, 'year'])
        if ys:
            terms.append(f"(series = '{series}' AND year IN ({', '.join(map(str, ys))}))")
    return ' OR '.join(terms) or None


def compute_add06(comp):
    """Fatias de CHUNK_CELLS celulas por faixa de latitude: densifica so a fatia e aplica o ADD-06.
    Cada fatia fica em PART/add06/ com marcador e assinatura (retoma). Devolve os parciais da execucao."""
    cond = add06_condition(comp)
    if cond is None:
        return []
    validity = SINK.read('l1_meta_month_validity').merge(
        comp[comp['series'].isin(ADD06_SERIES)], on=['series', 'year'], how='inner').reset_index(drop=True)
    source = table_source(STATS_MONTHLY)
    fingerprint = dict(monthly=table_fingerprint(STATS_MONTHLY), validity=table_fingerprint('l1_meta_month_validity'))
    pairs = sorted(f'{s}:{y}' for s, y in zip(comp['series'], comp['year']) if s in ADD06_SERIES)
    bands = query(f'SELECT DISTINCT lat_band FROM {source} WHERE {cond} ORDER BY 1')['lat_band'].tolist()
    produced = []
    for band in bands:
        band_cells = query(f'SELECT DISTINCT cell FROM {source} WHERE lat_band = {int(band)} AND ({cond}) '
                           'ORDER BY cell')['cell'].to_numpy(dtype='int64')
        for k, start in enumerate(range(0, len(band_cells), int(CHUNK_CELLS))):
            chunk = band_cells[start:start + int(CHUNK_CELLS)]
            part, marker = PART / 'add06' / f'b{int(band)}_{k}.parquet', PART / 'add06' / f'b{int(band)}_{k}.json'
            signature = dict(schema=CACHE_SCHEMA, grid_deg=GRID_DEG, stats_grid_deg=STATS_GRID_DEG,
                             max_unknown_days=MAX_UNKNOWN_DAYS, chunk_cells=int(CHUNK_CELLS), series_years=pairs,
                             band=int(band), k=k, first=int(chunk[0]), last=int(chunk[-1]), n_cells=len(chunk),
                             l1=fingerprint)
            produced.append(part)
            info = read_json(marker)
            if not FORCE and isinstance(info, dict) and info.get('signature') == signature and part.is_file():
                print(f'[reuse] add06 banda {int(band)} fatia {k}')
                continue
            marker.unlink(missing_ok=True)
            monthly = query(f'SELECT cell, year, month, series, cell_days FROM {source} WHERE lat_band = {int(band)} '
                            f'AND cell BETWEEN {int(chunk[0])} AND {int(chunk[-1])} AND ({cond})')
            monthly = monthly.astype({'cell': 'int64', 'year': 'int64', 'month': 'int64', 'cell_days': 'int64'})
            dense = densify_monthly(monthly, validity, chunk)
            result = add06_union_step(dense, MAX_UNKNOWN_DAYS)
            atomic_part(prepare_table(result, STATS_GRID_DEG, ['year']), part)
            atomic_json(dict(signature=signature, n_cells=len(chunk), rows=len(result)), marker)
            print(f'[add06] banda {int(band)} fatia {k}: {len(chunk)} celulas, {len(result)} linhas')
            del monthly, dense, result
    return produced

In [ ]:
# cell: run
selected_sensors = select_sensors()
files = index_core_files()
todo = sorted(k for k in files if k[0] in selected_sensors and (YEARS is None or k[1] in YEARS))
print(f'{len(files)} core files in processed/, {len(todo)} selected')
if not todo:
    raise ValueError(f'nenhum ficheiro selecionado (SENSORS={SENSORS}, YEARS={YEARS}); '
                     f'available core files in processed/: {sorted(files)}')
years = sorted({y for _, y in todo})
if YEARS is not None and set(YEARS) - set(years):
    raise FileNotFoundError(f'No selected source for years {sorted(set(YEARS) - set(years))}')

for sensor, year in todo:
    if is_done(sensor, year, files[(sensor, year)]) and not FORCE:
        print(f'[skip] {sensor} {year}')
        continue
    print(f'[process] {sensor} {year}')
    print('   ', process_file(sensor, year, files[(sensor, year)]))

selected_dirs = [PART / f'{s}_{y}' for s, y in todo]
selected_info = [read_json(d / '_done.json') for d in selected_dirs]
assert {info['grid_deg'] for info in selected_info} <= {GRID_DEG}, \
    'parciais calculados com grelhas diferentes; corra com FORCE = True'

# cell_year por ano: uma cache pertence a coorte exata das suas fontes, mesmo que haja outras mais novas
computed = set()
cell_year_files = []
for year in years:
    done, series = year_status(year, files, todo)
    computed |= {(s, year) for s in series}
    markers = [PART / f'{s}_{year}' / '_done.json' for s in done]
    provenance = dict(schema=CACHE_SCHEMA, grid_deg=GRID_DEG, series=sorted(series),
                      sources=[read_json(m)['signature'] for m in markers])
    target, meta = PART / f'cell_year_{year}.parquet', PART / f'cell_year_{year}.json'
    cell_year_files.append(target)
    if FORCE or read_json(meta) != provenance or not target.is_file():
        meta.unlink(missing_ok=True)
        cell_days = pd.concat([pd.read_parquet(m.parent / 'cell_days.parquet') for m in markers], ignore_index=True)
        cy = build_cell_year(cell_days)
        cy = cy[cy['series'].isin(series)].reset_index(drop=True)
        # Persist typed empty years too, replacing stale results without deleting source data.
        atomic_part(prepare_table(cy, GRID_DEG, ['series', 'year']), target)
        atomic_json(provenance, meta)
        print(f'[cell_year] {year}: {len(markers)} sensors, series {sorted(series)}, {len(cy)} rows')
        del cell_days, cy
    else:
        print(f'[skip] cell_year {year}')
comp = pd.DataFrame(sorted(computed), columns=['series', 'year']).astype({'series': 'object', 'year': 'int64'})

# ADD-06 por fatias de celulas, a partir do calendario mensal do L1
add06_files = compute_add06(comp)

In [ ]:
# cell: publish
published = {}

# --- por celula: cell_year e ADD-05 (consolidados com DuckDB a partir dos parciais) ---
published['l5_cell_year'] = consolidate('l5_cell_year', cell_year_files, ['cell', 'series', 'year'])
add05_files = [d / 'add05.parquet' for d in selected_dirs]
published['l5_add05_type_sensitivity'] = consolidate(
    'l5_add05_type_sensitivity', add05_files, ['cell', 'year', 'instrument', 'variant'],
    global_rows=add05_global_rows(add05_files))

# --- ESP-02/03/06: populacao, sobre TODAS as celulas de cell_year de uma vez ---
cell_year = pd.concat([pd.read_parquet(p, columns=['cell', 'series', 'year', 'n_days']) for p in cell_year_files],
                      ignore_index=True)
year_val = SINK.read('l1_meta_year_validity')
ok = (~year_val['partial'].astype(bool)) & (year_val['n_unknown_days'] <= MAX_UNKNOWN_DAYS)
# so entram (serie, ano) realmente calculados: numa execucao parcial os outros anos nao contam
eligible_years = (year_val.loc[ok, ['series', 'year']].astype({'series': 'object', 'year': 'int64'})
                  .merge(comp, on=['series', 'year']).reset_index(drop=True))
for name, tbl in {'l5_esp02_active_cells': esp02_active_cells(cell_year),
                  'l5_esp03_recurrence': esp03_recurrence(cell_year),
                  'l5_esp06_interannual': esp06_interannual(cell_year, eligible_years)}.items():
    write_table(SINK, name, tbl, grid_deg=GRID_DEG, partition_by=[])        # populacao (cell = -1): tabela pequena
    published[name] = len(tbl)
cell_years_active = esp06_cell_years_active(cell_year, eligible_years)      # por celula: particoes por defeito
write_table(SINK, 'l5_esp06_cell_years_active', cell_years_active, grid_deg=GRID_DEG)
published['l5_esp06_cell_years_active'] = len(cell_years_active)
del cell_year

# --- QA-05 (global): cobertura do L1 so das datas dos instrumentos/anos selecionados ---
coverage = SINK.read('l1_cal_coverage')
instrument_years = {('MODIS' if s == 'modis' else 'VIIRS', y) for s, y in todo}
coverage_years = pd.to_datetime(coverage['date']).dt.year
coverage = coverage.loc[[(i, y) in instrument_years for i, y in zip(coverage['instrument'], coverage_years)]]
qa05 = qa05_confirmed_days(coverage)
write_table(SINK, 'l5_qa05_confirmed_days', qa05, grid_deg=GRID_DEG, partition_by=[])
published['l5_qa05_confirmed_days'] = len(qa05)

# --- ADD-06: linha global = soma entre fatias (sum_modis, sum_combined); o racio recalcula-se; n_months e a uniao ---
if add06_files:
    glob = query(f"SELECT year, max(n_months)::BIGINT AS n_months, sum(sum_modis)::BIGINT AS sum_modis, "
                 f"sum(sum_combined)::BIGINT AS sum_combined FROM read_parquet({duckdb_list(add06_files)}, "
                 f"union_by_name=true) WHERE cell = -1 GROUP BY year ORDER BY year")
    ratio = glob['sum_combined'] / glob['sum_modis'].where(glob['sum_modis'] != 0)
    glob['ratio_combined_modis'] = ratio.astype('float64')
    glob['excess_pct'] = (100.0 * (ratio - 1.0)).astype('float64')
    glob['cell'] = np.full(len(glob), common.GLOBAL_CELL, dtype='int64')
    glob['year'] = glob['year'].astype('int64')
    glob = prepare_table(glob[['cell', 'year', 'n_months', 'sum_modis', 'sum_combined',
                               'ratio_combined_modis', 'excess_pct']], STATS_GRID_DEG, ['year'])
    published['l5_add06_union_step'] = consolidate('l5_add06_union_step', add06_files, ['cell', 'year'], global_rows=glob)
else:
    empty = empty_add06()
    write_table(SINK, 'l5_add06_union_step', empty, grid_deg=STATS_GRID_DEG, partition_by=[])
    published['l5_add06_union_step'] = 0
for name, n in published.items():
    print(f'{name}: {n} linhas')

try:
    commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ''
# _runs escreve-se POR ULTIMO: uma execucao que parou a meio nao fica registada
write_run(SINK, 'lote5', commit=commit or 'unknown', grid_deg=GRID_DEG, tables=published, params={
    'sensors': sorted(selected_sensors), 'years': years, 'selected_files': [files[k].name for k in todo],
    'grid_deg': GRID_DEG, 'stats_grid_deg': STATS_GRID_DEG, 'max_unknown_days': MAX_UNKNOWN_DAYS,
    'chunk_cells': int(CHUNK_CELLS), 'series_years_computed': [f'{s}:{y}' for s, y in sorted(computed)],
    'eligible_type': common.ELIGIBLE_TYPE, 'eligible_confidence': list(common.ELIGIBLE_CONFIDENCE),
    'sources': [{k: v for k, v in info.items() if k != 'signature'} for info in selected_info],
    'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')})
print(pd.DataFrame({'table': list(published), 'n_rows': list(published.values())}).to_string(index=False))

## Verificação
Confirma invariantes entre as tabelas e contra o L1 (falha com `AssertionError` se alguma não bater): ESP-02 = `n_active_cells` do ESP-03, baldes do ESP-03, `cell_year` célula a célula contra o CAL-04 (`l1_cal_annual`), linhas globais do ADD-05/ADD-06 = soma das células, ADD-05 `default` = células-dia do `cell_year`.

In [ ]:
# cell: verify
SINK_NAMES = ['l5_cell_year', 'l5_esp02_active_cells', 'l5_esp03_recurrence', 'l5_esp06_interannual',
              'l5_esp06_cell_years_active', 'l5_add05_type_sensitivity', 'l5_qa05_confirmed_days', 'l5_add06_union_step']
assert all(SINK.exists(n) for n in SINK_NAMES), 'faltam tabelas L5 publicadas'
keys = ['cell', 'series', 'year']
e2 = SINK.read('l5_esp02_active_cells').sort_values(keys).reset_index(drop=True)
e3 = SINK.read('l5_esp03_recurrence').sort_values(keys).reset_index(drop=True)
assert e2[keys + ['n_active_cells']].equals(e3[keys + ['n_active_cells']]), 'ESP-02 != n_active_cells do ESP-03'
assert (e2['cell'] == common.GLOBAL_CELL).all()
buckets = e3[['n_cells_1d', 'n_cells_2_5d', 'n_cells_6_15d', 'n_cells_16plus']].sum(axis=1)
assert (buckets == e3['n_active_cells']).all(), 'baldes de recorrencia nao somam n_active_cells'
e6 = SINK.read('l5_esp06_interannual')
assert (e6['n_cells_2plus_years'] <= e6['n_cells']).all()
assert e6['pct_cells_2plus'].dropna().between(0, 100).all()
q5 = SINK.read('l5_qa05_confirmed_days')
assert q5['pct_confirmed'].between(0, 100).all() and (q5['n_days_confirmed'] <= q5['n_days_expected']).all()
a6 = SINK.read('l5_add06_union_step')
assert (a6['ratio_combined_modis'].dropna() >= 1).all(), 'a uniao nunca pode ter menos celulas-dia que o MODIS'

# ESP-02 por (serie, ano) = celulas de cell_year; cell_year == CAL-04 celula a celula (series/anos calculados)
cy_totals = query(f"SELECT series, year, count(*)::BIGINT AS n_cells, sum(n_days)::BIGINT AS n_days "
                  f"FROM {table_source('l5_cell_year')} GROUP BY series, year")
merged = e2.merge(cy_totals, on=['series', 'year'], how='outer')
assert (merged['n_active_cells'].fillna(0) == merged['n_cells'].fillna(0)).all(), 'ESP-02 != celulas de cell_year'
l1_series = set(query(f"SELECT DISTINCT series FROM {table_source('l1_cal_annual')}")['series'])
terms = [f"(series = '{s}' AND year = {int(y)})" for s, y in sorted(computed) if s in l1_series]
cond = ' OR '.join(terms) or 'FALSE'
diff = query(f"WITH cy AS (SELECT cell, series, year, n_days FROM {table_source('l5_cell_year')} WHERE {cond}), "
             f"an AS (SELECT cell, series, year, cell_days FROM {table_source('l1_cal_annual')} WHERE {cond}) "
             f"SELECT count(*)::BIGINT AS n, coalesce(sum((cy.n_days IS DISTINCT FROM an.cell_days)::INTEGER), 0)::BIGINT AS n_diff "
             f"FROM cy FULL OUTER JOIN an USING (cell, series, year)")
assert int(diff['n_diff'].iloc[0]) == 0, 'cell_year != CAL-04 (l1_cal_annual) em alguma celula'

# ADD-05: global = soma das celulas; `default` = celulas-dia do cell_year da serie do instrumento
a5 = SINK.read('l5_add05_type_sensitivity')
cells5, glob5 = a5[a5['cell'] != common.GLOBAL_CELL], a5[a5['cell'] == common.GLOBAL_CELL]
sums = (cells5.groupby(['year', 'instrument', 'variant'])[['n_eligible', 'n_cell_days']].sum().reset_index())
check5 = glob5.merge(sums, on=['year', 'instrument', 'variant'], suffixes=('', '_sum'), how='outer')
assert (check5['n_eligible'] == check5['n_eligible_sum']).all() and (check5['n_cell_days'] == check5['n_cell_days_sum']).all(), \
    'ADD-05 global != soma das celulas'
file_globals = pd.concat([pd.read_parquet(d / 'add05_global.parquet') for d in selected_dirs], ignore_index=True)
file_sums = file_globals.groupby(['year', 'instrument', 'variant'])[['n_eligible', 'n_cell_days']].sum().reset_index()
pd.testing.assert_frame_equal(
    glob5[['year', 'instrument', 'variant', 'n_eligible', 'n_cell_days']].astype({'instrument': 'object', 'variant': 'object'})
    .sort_values(['year', 'instrument', 'variant']).reset_index(drop=True),
    file_sums.astype({'instrument': 'object', 'variant': 'object'}).sort_values(['year', 'instrument', 'variant']).reset_index(drop=True),
    check_dtype=False)
default5 = glob5[glob5['variant'] == 'default']
for row in default5.itertuples():
    series = 'modis' if row.instrument == 'MODIS' else 'viirs'
    if (series, row.year) in computed:
        expected = cy_totals[(cy_totals['series'] == series) & (cy_totals['year'] == row.year)]['n_days'].sum()
        assert row.n_cell_days == expected, f'ADD-05 default {row.instrument} {row.year} != cell_year'

# ADD-06: global = soma das celulas por ano; n_months = n_months de cada celula (a validade e global)
cells6, glob6 = a6[a6['cell'] != common.GLOBAL_CELL], a6[a6['cell'] == common.GLOBAL_CELL]
by_year = cells6.groupby('year')[['sum_modis', 'sum_combined']].sum().reset_index()
check6 = glob6.merge(by_year, on='year', suffixes=('', '_sum'), how='outer')
assert (check6['sum_modis'] == check6['sum_modis_sum']).all() and (check6['sum_combined'] == check6['sum_combined_sum']).all(), \
    'ADD-06 global != soma das celulas'
if len(cells6):
    assert (cells6['n_months'] <= cells6['year'].map(glob6.set_index('year')['n_months'])).all()
print(f"OK: invariantes verificadas ({len(cy_totals)} (serie, ano) em cell_year, {int(diff['n'].iloc[0])} celula-serie-ano "
      f"comparadas com o CAL-04)")